# Section 05b Onboarding Notebook: Special Correlated Methods — Bootstrap Embedding (BE) & Density Matrix Renormalization Group (DMRG)

### Overview & Pedagogical Design
Standard Coupled Cluster (`CCSD(T)`) scales as $\mathcal{O}(N^7)$ and exact Full CI scales exponentially with system size, limiting their application to small molecules. Modern quantum chemistry overcomes this bottleneck via two complementary paradigms that exploit **quantum entanglement locality**:
1. **Bootstrap Embedding (BE)** ([LargeBasisBE.pdf](https://drive.google.com/file/d/11nN9UOAku31tvnLTWJ3peB9QWuAi8jSb/view?usp=drivesdk); [J. Chem. Theory Comput. 2024](https://pubs.acs.org/doi/10.1021/acs.jctc.4c01267); [QuEmb](https://github.com/troyvvgroup/quemb)): Decomposes a large molecular system into overlapping local fragments $\mathbb{F}_A$ in an orthonormal Localized Orbital (LO) basis. By applying **Schmidt decomposition (SVD)** to the mean-field occupied orbital coefficients $\mathbf{C}_{\text{occ}}$, BE compresses the entire surrounding environment $\bar{\mathbb{F}}_A$ into at most $N_{\text{frag}}$ entangled **Bath Orbitals (BOs)**, solves each $2N_{\text{frag}}$-orbital impurity problem with a high-level correlated solver, matches edge-to-center densities across overlapping fragments via a Newton–Raphson potential loop, and reconstructs the total energy using the **two-body cumulant functional**.
2. **Density Matrix Renormalization Group (DMRG)**: Represents strongly correlated 1D/quasi-1D wavefunctions as a **Matrix Product State (MPS)** tensor network $|\Psi\rangle = \sum_{\{s_\ell\}} A^{[1]s_1} A^{[2]s_2} \cdots A^{[L]s_L} |s_1 s_2 \dots s_L\rangle$ and variationally optimizes two adjacent site tensors at a time under a **Matrix Product Operator (MPO)** representation of the Hamiltonian.

In this notebook, **no BE or DMRG functions are pre-solved for you**. You will implement **16 granular `# TODO` functions** covering every mathematical and algorithmic step of both methods:
- **Part 1 — Bootstrap Embedding: Schmidt Bath Decomposition & Active-Space Projection (`TODO 1` – `TODO 4`)**
- **Part 2 — Bootstrap Embedding: Dressed Impurity Solver, Cumulant Energy & Newton–Raphson Matching (`TODO 5` – `TODO 9`)**
- **Part 3 — Matrix Product States (MPS), Mixed-Canonical Gauge & MPO Construction (`TODO 10` – `TODO 12`)**
- **Part 4 — Two-Site Variational DMRG Environments, Effective Hamiltonian & Sweeps (`TODO 13` – `TODO 16`)**
- **Part 5 — Step-by-Step Checkpoint & Verification Suite (`run_section05b_checkpoints`)**

In [ ]:
import numpy as np
from scipy.linalg import eigh, svd, qr
from pyscf import gto, scf, lo
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)

# Build a 4-atom Hydrogen ring/chain in STO-3G directly with PySCF and transform to Löwdin Orthogonalized AOs (LOs)
mol_h4 = gto.M(
    atom="H 0.0 0.0 0.0; H 0.0 0.0 1.5; H 1.5 0.0 1.5; H 1.5 0.0 0.0",
    unit="Bohr",
    basis="sto-3g",
    verbose=0,
)
mf_h4 = scf.RHF(mol_h4).run(conv_tol=1e-12)
S_ao = mol_h4.intor("int1e_ovlp")
evals_s, evecs_s = eigh(S_ao)
X_lowdin = (evecs_s / np.sqrt(evals_s)) @ evecs_s.T      # S^(-1/2): AO -> Orthonormal LO basis
X_lowdin_inv = (evecs_s * np.sqrt(evals_s)) @ evecs_s.T  # S^(+1/2): MO coeff -> Orthonormal LO basis

h_lo_h4 = X_lowdin.T @ mf_h4.get_hcore() @ X_lowdin
V_lo_h4 = np.einsum("uvls,up,vq,lr,st->pqrt", mol_h4.intor("int2e"), X_lowdin, X_lowdin, X_lowdin, X_lowdin, optimize=True)
C_occ_lo_h4 = X_lowdin_inv @ mf_h4.mo_coeff[:, :2]

print("Initialized PySCF H4 (STO-3G) in Löwdin orthonormalized LO basis. Implement TODOs 1-16 and run `run_section05b_checkpoints()`!")

## Part 1: Bootstrap Embedding — Schmidt Decomposition & Active-Space Projection (TODOs 1–4)

### Mathematical Specifications
Given an $N_{\text{LO}}$-orbital orthonormal Localized Orbital (LO) basis ($S_{\mu\nu}^{\text{LO}} = \delta_{\mu\nu}$) and a converged full-system RHF occupied coefficient matrix $\mathbf{C}_{\text{occ}} \in \mathbb{R}^{N_{\text{LO}} \times n_{\text{occ}}}$:
1. **Schmidt SVD on the Complement Block (`partition_c_occ_and_schmidt_svd` — TODO 1)**:
   For fragment $A$ with LO indices $\mathbb{F}_A$ (`frag_indices`) of size $N_{\text{frag}}$ and environment complement indices $\bar{\mathbb{F}}_A = \{0, \dots, N_{\text{LO}}-1\} \setminus \mathbb{F}_A$ (`comp_indices`), slice the complement rows $\mathbf{C}_{\text{comp}} = \mathbf{C}_{\text{occ}}[\bar{\mathbb{F}}_A, :]$ and compute its Singular Value Decomposition ([LargeBasisBE.pdf](https://drive.google.com/file/d/11nN9UOAku31tvnLTWJ3peB9QWuAi8jSb/view?usp=drivesdk)):
   $$\mathbf{C}_{\text{comp}} = \mathbf{U}_{\text{comp}}\,\text{diag}(\boldsymbol{\sigma})\,\mathbf{V}^\mathsf{T}$$
   Because $\mathbf{C}_{\text{occ}}^\mathsf{T}\mathbf{C}_{\text{occ}} = \mathbf{I}_{n_{\text{occ}}}$, the singular values satisfy $\sigma_k \in [0, 1]$:
   - Columns of $\mathbf{U}_{\text{comp}}$ with $\text{svd\_tol} < \sigma_k < 1 - \text{svd\_tol}$ are **entangled Bath Orbitals ($\mathbf{U}_{\text{bath}}$)** (at most $N_{\text{frag}}$ columns).
   - Columns of $\mathbf{U}_{\text{comp}}$ with $\sigma_k \ge 1 - \text{svd\_tol}$ are **pure doubly-occupied Environment Orbitals ($\mathbf{U}_{\text{env}}$)** unentangled with $\mathbb{F}_A$.

2. **Embedding Transformation $\mathbf{T}^{(A)}$ & Frozen Environment 1-RDM (`assemble_embedding_rotation_and_env_density` — TODO 2)**:
   Form the $(N_{\text{LO}} \times N_{\text{emb}})$ isometric embedding matrix $\mathbf{T}^{(A)}$ (where $N_{\text{emb}} = N_{\text{frag}} + N_{\text{bath}}$):
   - The first $N_{\text{frag}}$ columns place the identity matrix $\mathbf{I}_{N_{\text{frag}}}$ on the rows `frag_indices`.
   - The remaining $N_{\text{bath}}$ columns place $\mathbf{U}_{\text{bath}}$ on the rows `comp_indices`.
   - The frozen pure-environment spin-summed 1-RDM in the full LO basis is $\mathbf{P}_{\text{env}}^{\text{LO}} = 2\,\mathbf{U}_{\text{env,full}}\,\mathbf{U}_{\text{env,full}}^\mathsf{T}$ (supported on `comp_indices`).

3. **Active-Space Hamiltonian Projection (`project_hamiltonian_to_embedding_space` — TODO 3)**:
   The frozen environment electrons screen the fragment via the Coulomb–Exchange core potential:
   $$G_{\mu\nu}[\mathbf{P}_{\text{env}}^{\text{LO}}] = \sum_{\lambda\sigma} P_{\text{env},\lambda\sigma}^{\text{LO}}\left[ (\mu\nu|\lambda\sigma)_{\text{LO}} - \frac{1}{2}(\mu\lambda|\nu\sigma)_{\text{LO}} \right], \qquad \mathbf{h}_{\text{eff}}^{(A)} = \mathbf{h}_{\text{LO}} + \mathbf{G}[\mathbf{P}_{\text{env}}^{\text{LO}}]$$
   Project both 1-electron and 2-electron integrals into the $N_{\text{emb}}$-orbital active embedding space:
   $$\mathbf{h}_{\text{emb}}^{(A)} = \mathbf{T}^{(A)\mathsf{T}}\,\mathbf{h}_{\text{eff}}^{(A)}\,\mathbf{T}^{(A)}, \qquad V_{\text{emb}, pqrs}^{(A)} = \sum_{\mu\nu\lambda\sigma} T_{\mu p}^{(A)} T_{\nu q}^{(A)} T_{\lambda r}^{(A)} T_{\sigma s}^{(A)}\,V_{\mu\nu\lambda\sigma}^{\text{LO}}$$

4. **1-RDM Off-Diagonal Block SVD (`extract_1rdm_schmidt_bath` — TODO 4)**:
   Equivalently, given the full-system idempotent 1-RDM $\mathbf{P}_{\text{LO}} = 2\mathbf{C}_{\text{occ}}\mathbf{C}_{\text{occ}}^\mathsf{T}$, taking the SVD of the off-diagonal coupling block $\mathbf{P}_{\text{env,frag}} = \mathbf{P}_{\text{LO}}[\bar{\mathbb{F}}_A, \mathbb{F}_A] = \mathbf{U}\,\boldsymbol{\Sigma}\,\mathbf{V}^\mathsf{T}$ directly yields the bath orbitals as the left singular vectors $\mathbf{U}_{:, \Sigma_k > \text{tol}}$!

In [ ]:
def partition_c_occ_and_schmidt_svd(
    C_occ_lo: np.ndarray, frag_indices: list[int], svd_tol: float = 1e-7
) -> dict[str, np.ndarray | list[int] | int]:
    '''
    TODO 1: Partition C_occ_lo into fragment rows (frag_indices) and complement rows (comp_indices),
    perform an SVD on C_comp = C_occ_lo[comp_indices, :], and classify columns of U_comp into
    entangled bath orbitals (svd_tol < s < 1 - svd_tol) and frozen environment orbitals (s >= 1 - svd_tol).

    Returns
    -------
    dict with keys:
        'comp_indices': list[int]
        'singular_values': np.ndarray
        'U_bath': np.ndarray of shape (n_comp, n_bath)
        'U_env': np.ndarray of shape (n_comp, n_env)
        'n_bath': int
        'n_env': int
    '''
    # TODO 1: Compute comp_indices, slice C_comp, call scipy.linalg.svd(C_comp, full_matrices=False),
    #         and mask singular values into bath_mask and env_mask.
    raise NotImplementedError("Complete TODO 1: partition_c_occ_and_schmidt_svd")


def assemble_embedding_rotation_and_env_density(
    n_lo: int,
    frag_indices: list[int],
    comp_indices: list[int],
    U_bath: np.ndarray,
    U_env: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 2: Construct the (n_lo, n_frag + n_bath) isometric embedding matrix T_emb
    and the (n_lo, n_lo) frozen pure-environment spin-summed 1-RDM P_env_lo.
    Return (T_emb, P_env_lo).
    '''
    # TODO 2a: Allocate T_emb of shape (n_lo, n_frag + n_bath), place identity on frag_indices[:, :n_frag],
    #          and place U_bath on comp_indices[:, n_frag:].
    # TODO 2b: Form U_env_full of shape (n_lo, n_env) on comp_indices and return P_env_lo = 2.0 * U_env_full @ U_env_full.T.
    raise NotImplementedError("Complete TODO 2: assemble_embedding_rotation_and_env_density")


def project_hamiltonian_to_embedding_space(
    h_lo: np.ndarray,
    V_lo: np.ndarray,
    T_emb: np.ndarray,
    P_env_lo: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 3: Compute the core-environment HF potential G[P_env_lo] = J[P_env_lo] - 0.5 * K[P_env_lo],
    form h_eff_lo = h_lo + G[P_env_lo], and project both h_eff_lo and V_lo into the embedding basis
    using T_emb. Return (h_emb, V_emb).
    '''
    # TODO 3a: Contract V_lo with P_env_lo via np.einsum to form J_env and K_env, and set h_eff = h_lo + J_env - 0.5 * K_env.
    # TODO 3b: Transform h_emb = T_emb.T @ h_eff @ T_emb and 4-index V_emb via O(N^5) np.einsum.
    raise NotImplementedError("Complete TODO 3: project_hamiltonian_to_embedding_space")


def extract_1rdm_schmidt_bath(
    P_lo: np.ndarray, frag_indices: list[int], svd_tol: float = 1e-8
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 4: Extract entangled bath orbitals directly from the off-diagonal coupling block
    P_env_frag = P_lo[np.ix_(comp_indices, frag_indices)] via SVD.
    Return (singular_values_kept, T_emb_from_1rdm).
    '''
    # TODO 4: Slice P_env_frag, compute its SVD, retain columns with singular_val > svd_tol, and build T_emb.
    raise NotImplementedError("Complete TODO 4: extract_1rdm_schmidt_bath")

## Part 2: Bootstrap Embedding — Impurity Solver, Cumulant Energy & Density Matching Loop (TODOs 5–9)

### Mathematical Specifications
1. **Dressed Fragment Impurity Solver (`solve_dressed_fragment_fci` — TODO 5)**:
   In fragment $A$'s embedding basis, let `center_indices_in_frag` ($\mathbb{C}_A$) and `edge_indices_in_frag` ($\mathbb{E}_A$) denote the local indices of the center and edge orbitals. To enforce global electron conservation ($\mu$) and edge-to-center density matching ($\boldsymbol{\lambda}^{(A)}$), dress the 1-electron embedding Hamiltonian:
   $$\tilde{h}_{\text{emb}, pp}^{(A)} = h_{\text{emb}, pp}^{(A)} - \mu \quad (p \in \mathbb{C}_A), \qquad \tilde{h}_{\text{emb}, ee}^{(A)} = h_{\text{emb}, ee}^{(A)} + \lambda_e^{(A)} \quad (e \in \mathbb{E}_A)$$
   Then solve the active-space 2-electron singlet Full CI eigenvalue problem for $(\tilde{\mathbf{h}}_{\text{emb}}^{(A)}, \mathbf{V}_{\text{emb}}^{(A)})$ to obtain the correlated spin-summed 1-RDM $\mathbf{P}_{\text{corr}}^{(A)}$ and 2-RDM $\boldsymbol{\Gamma}_{\text{corr}}^{(A)}$.

2. **Two-Body Cumulant Tensor (`compute_two_body_cumulant_and_delta_p` — TODO 6)**:
   Given the correlated $\mathbf{P}_{\text{corr}}^{(A)}, \boldsymbol{\Gamma}_{\text{corr}}^{(A)}$ and the fragment HF 1-RDM $\mathbf{P}_{\text{HF}}^{(A)}$ ([JCTC 2024](https://pubs.acs.org/doi/10.1021/acs.jctc.4c01267)):
   $$\Delta\mathbf{P}^{(A)} = \mathbf{P}_{\text{corr}}^{(A)} - \mathbf{P}_{\text{HF}}^{(A)}, \qquad \tilde{K}_{pqrs}^{(A)} = \Gamma_{\text{corr}, pqrs}^{(A)} - \left(P_{\text{corr}, pq}^{(A)} P_{\text{corr}, rs}^{(A)} - \frac{1}{2} P_{\text{corr}, ps}^{(A)} P_{\text{corr}, rq}^{(A)}\right)$$
   Under the standard BE approximate cumulant (`use_be_approx=True`), add the quadratic $\Delta\mathbf{P}$ correction so that the normal-ordered mean-field reference is exact:
   $$K_{pqrs}^{\text{BE},(A)} = \tilde{K}_{pqrs}^{(A)} + \left(\Delta P_{pq}^{(A)}\Delta P_{rs}^{(A)} - \frac{1}{2}\Delta P_{ps}^{(A)}\Delta P_{rq}^{(A)}\right)$$

3. **Fragment Cumulant Correlation Energy (`compute_fragment_cumulant_energy_contribution` — TODO 7)**:
   Using the **undressed** fragment HF Fock matrix $\mathbf{F}_{\text{HF,emb}}^{(A)} = \mathbf{T}^{(A)\mathsf{T}}\mathbf{F}_{\text{HF}}^{\text{LO}}\mathbf{T}^{(A)}$, sum **only over the center orbitals $p \in \mathbb{C}_A$** so overlapping fragments never double-count:
   $$\Delta E_A^{\text{corr}} = \sum_{p \in \mathbb{C}_A} \left[ \sum_{q=1}^{N_{\text{emb}}} F_{\text{HF,emb}, pq}^{(A)}\,\Delta P_{pq}^{(A)} + \frac{1}{2}\sum_{q,r,s=1}^{N_{\text{emb}}} V_{\text{emb}, pqrs}^{(A)}\,K_{pqrs}^{(A)} \right]$$

4. **Density Matching Residuals & Newton–Raphson Loop (`evaluate_be_matching_residuals` & `run_bootstrap_embedding_newton_loop` — TODOs 8–9)**:
   For each overlapping edge site $e \in \mathbb{E}_A$ on fragment $A$ that corresponds to a center site $c \in \mathbb{C}_B$ on neighboring fragment $B$, require $P_{ee}^{(A)}(\boldsymbol{\lambda}, \mu) - P_{cc}^{(B)}(\boldsymbol{\lambda}, \mu) = 0$, alongside the global center-electron condition $\big(\sum_A \sum_{p \in \mathbb{C}_A} P_{pp}^{(A)}(\boldsymbol{\lambda}, \mu)\big) - N_{\text{elec}} = 0$. Solve $\mathbf{R}(\boldsymbol{\Theta}) = \mathbf{0}$ using finite-difference Newton–Raphson updates $\mathbf{J}\,\delta\boldsymbol{\Theta} = -\mathbf{R}$.

In [ ]:
def solve_dressed_fragment_fci(
    h_emb: np.ndarray,
    V_emb: np.ndarray,
    center_indices_in_frag: list[int],
    edge_indices_in_frag: list[int],
    mu_global: float = 0.0,
    lambda_edges: np.ndarray | None = None,
) -> tuple[float, np.ndarray, np.ndarray]:
    '''
    TODO 5: Dress h_emb with -mu_global on center_indices_in_frag and +lambda_edges on edge_indices_in_frag,
    solve the 2-electron singlet Full CI problem on (h_dressed, V_emb), and return
    (e_elec_dressed, rdm1_corr, rdm2_corr).
    '''
    # TODO 5: Copy h_emb, apply -mu_global to diagonal center_indices_in_frag and +lambda_edges to diagonal
    #         edge_indices_in_frag, build the (n_emb^2, n_emb^2) 2e singlet Hamiltonian, diagonalize with eigh,
    #         symmetrize/normalize C_mat, and return (evals[0], 2 * C @ C.T, 2 * einsum("pr,qs->pqrs", C, C)).
    raise NotImplementedError("Complete TODO 5: solve_dressed_fragment_fci")


def compute_two_body_cumulant_and_delta_p(
    rdm1_corr: np.ndarray,
    rdm2_corr: np.ndarray,
    rdm1_hf: np.ndarray,
    use_be_approx: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 6: Compute delta_p = rdm1_corr - rdm1_hf and the 4-index two-body cumulant K_{pqrs}.
    Return (delta_p, k2_cumulant).
    '''
    # TODO 6: Compute k_tilde = rdm2_corr - (einsum("pq,rs->pqrs", rdm1_corr, rdm1_corr) - 0.5 * einsum("ps,rq->pqrs", rdm1_corr, rdm1_corr)).
    #         If use_be_approx is True, add the quadratic delta_p terms.
    raise NotImplementedError("Complete TODO 6: compute_two_body_cumulant_and_delta_p")


def compute_fragment_cumulant_energy_contribution(
    center_indices_in_frag: list[int],
    F_hf_emb_undressed: np.ndarray,
    V_emb: np.ndarray,
    delta_p: np.ndarray,
    k2_cumulant: np.ndarray,
) -> float:
    '''
    TODO 7: Evaluate the cumulant correlation energy contribution dE_A of fragment A
    by contracting only over center rows p in center_indices_in_frag.
    '''
    # TODO 7: Slice c_idx = center_indices_in_frag and return
    #         einsum("pq,pq->", F_hf_emb_undressed[c_idx, :], delta_p[c_idx, :])
    #         + 0.5 * einsum("pqrs,pqrs->", V_emb[c_idx, :, :, :], k2_cumulant[c_idx, :, :, :]).
    raise NotImplementedError("Complete TODO 7: compute_fragment_cumulant_energy_contribution")


def evaluate_be_matching_residuals(
    fragment_rdms: list[np.ndarray],
    matching_pairs: list[tuple[int, int, int, int]],
    center_indices_in_frag_list: list[list[int]],
    target_n_elec: int,
) -> np.ndarray:
    '''
    TODO 8: Assemble the residual vector R = [..., P^(A)_{ee} - P^(B)_{cc}, ..., total_center_elec - target_n_elec].
    Here each tuple in `matching_pairs` is (frag_A, local_edge_e, frag_B, local_center_c).
    '''
    # TODO 8: Evaluate edge-minus-center density differences for each tuple in matching_pairs,
    #         append the global electron count residual, and return as a 1D float64 numpy array.
    raise NotImplementedError("Complete TODO 8: evaluate_be_matching_residuals")


def run_bootstrap_embedding_newton_loop(
    h_lo: np.ndarray,
    V_lo: np.ndarray,
    e_nuc: float,
    n_elec: int,
    fragments: list[list[int]],
    centers: list[list[int]],
    max_iter: int = 15,
    rms_tol: float = 1e-9,
) -> dict[str, float | int | list[float] | np.ndarray]:
    '''
    TODO 9: Execute the complete Bootstrap Embedding workflow with Newton-Raphson density matching
    using TODOs 1-8. Return a dict with keys:
    'e_hf_total', 'e_corr_cumulant', 'e_be_total', 'fragment_energies_corr',
    'total_center_electrons', 'initial_rms_mismatch', 'final_rms_mismatch', 'n_iterations'.
    '''
    # TODO 9: 1. Run full-system RHF on (I, h_lo, 0, V_lo, e_nuc, n_elec) to get C_occ_lo, P_hf_lo, F_hf_lo.
    #         2. Build Schmidt spaces and projected (h_emb, V_emb, F_hf_emb, P_hf_emb) for every fragment.
    #         3. Identify matching_pairs (frag_A, edge_e, frag_B, center_c) and optimize [lambda_edges..., mu]
    #            via finite-difference Newton-Raphson until ||R||_RMS < rms_tol.
    #         4. Sum fragment cumulant correlation energies and return the result dictionary.
    raise NotImplementedError("Complete TODO 9: run_bootstrap_embedding_newton_loop")

## Part 3: Matrix Product States (MPS), Mixed-Canonical Gauge & MPO Construction (TODOs 10–12)

### Mathematical Specifications
1. **Exact Statevector to Left-Canonical MPS via Sequential SVDs (`wavefunction_to_mps_svd` — TODO 10)**:
   Given a $d^L$-component statevector $|\Psi\rangle$ on $L$ sites (local physical dimension $d = 2$), reshape and apply SVD sequentially from site $\ell = 0$ to $L-2$:
   $$\Psi_{(\alpha_{\ell-1} s_\ell),\, (s_{\ell+1}\dots s_{L-1})} = \sum_{\alpha_\ell=1}^{\chi_\ell} U_{(\alpha_{\ell-1} s_\ell), \alpha_\ell}\,\sigma_{\alpha_\ell}\,V^\dagger_{\alpha_\ell, (s_{\ell+1}\dots s_{L-1})}$$
   Reshape $\mathbf{U}$ into the left-normalized rank-3 tensor $A^{[\ell]}_{\alpha_{\ell-1}, s_\ell, \alpha_\ell}$ (satisfying $\sum_{s_\ell} A^{[\ell]s_\ell\dagger} A^{[\ell]s_\ell} = \mathbf{I}$), record the bipartite von Neumann entanglement entropy $S_{\text{vN}}(\ell) = -\sum_k p_k \ln p_k$ ($p_k = \sigma_k^2 / \sum_j \sigma_j^2$), and pass $\text{diag}(\boldsymbol{\sigma})\mathbf{V}^\dagger$ to the next site!

2. **Mixed-Canonical MPS Gauge Around a Target Bond (`canonicalize_mps_mixed_gauge` — TODO 11)**:
   In two-site DMRG optimizing bond `(site, site + 1)`, all tensors to the left ($0 \dots \text{site}-1$) must be **left-orthonormal** ($\sum_s A^{[j]s\dagger}A^{[j]s} = \mathbf{I}$ via QR decomposition) and all tensors to the right ($\text{site}+2 \dots L-1$) must be **right-orthonormal** ($\sum_s B^{[j]s}B^{[j]s\dagger} = \mathbf{I}$ via RQ/SVD decomposition). This ensures the effective environment basis is strictly orthonormal ($\mathbf{S}_{\text{eff}} = \mathbf{I}$), turning the generalized eigenvalue problem into a standard Hermitian eigenvalue problem!

3. **Matrix Product Operator (MPO) for the 1D Spin-$1/2$ XXZ Heisenberg Chain (`build_heisenberg_xxz_mpo` — TODO 12)**:
   For the 1D spin-$1/2$ XXZ Hamiltonian:
   $$\hat{H} = \sum_{\ell=0}^{L-2} \left[ \frac{J_{xy}}{2}\left(\hat{S}_\ell^+ \hat{S}_{\ell+1}^- + \hat{S}_\ell^- \hat{S}_{\ell+1}^+\right) + J_z\,\hat{S}_\ell^z \hat{S}_{\ell+1}^z \right] - h_z \sum_{\ell=0}^{L-1} \hat{S}_\ell^z$$
   each bulk MPO tensor $W^{[\ell]}_{b_{\ell-1}, b_\ell, s_\ell, s'_\ell}$ has shape `(5, 5, 2, 2)` corresponding to the operator-valued lower-triangular automaton matrix:
   $$W_{\text{bulk}} = \begin{pmatrix} \hat{I} & 0 & 0 & 0 & 0 \\ \hat{S}^+ & 0 & 0 & 0 & 0 \\ \hat{S}^- & 0 & 0 & 0 & 0 \\ \hat{S}^z & 0 & 0 & 0 & 0 \\ -h_z \hat{S}^z & \frac{J_{xy}}{2}\hat{S}^- & \frac{J_{xy}}{2}\hat{S}^+ & J_z \hat{S}^z & \hat{I} \end{pmatrix}$$
   with boundary tensors $W^{[0]} = W_{\text{bulk}}[4:5, :, :, :]$ of shape `(1, 5, 2, 2)` and $W^{[L-1]} = W_{\text{bulk}}[:, 0:1, :, :]$ of shape `(5, 1, 2, 2)`.

In [ ]:
def wavefunction_to_mps_svd(
    psi_vec: np.ndarray, n_sites: int, d_phys: int = 2, max_bond_dim: int = 16, svd_cutoff: float = 1e-12
) -> tuple[list[np.ndarray], np.ndarray]:
    '''
    TODO 10: Decompose a 2^n_sites statevector `psi_vec` into a list of rank-3 MPS tensors
    [A_0, ..., A_{n_sites-1}] of shape (chi_left, d_phys, chi_right) via sequential SVDs,
    and return (mps_tensors, bond_entropies).
    '''
    # TODO 10: Loop over bonds 0 .. n_sites-2, reshape remainder to (chi_l * d_phys, -1),
    #          compute SVD, truncate to max_bond_dim and svd_cutoff, compute S_vN = -sum(p log p),
    #          append U.reshape(chi_l, d_phys, chi_r), and propagate diag(s) @ Vh to the next site.
    raise NotImplementedError("Complete TODO 10: wavefunction_to_mps_svd")


def canonicalize_mps_mixed_gauge(
    mps_tensors: list[np.ndarray], left_site: int
) -> list[np.ndarray]:
    '''
    TODO 11: Return a copy of `mps_tensors` in mixed-canonical gauge around bond (left_site, left_site + 1),
    where sites 0 .. left_site - 1 are left-orthonormal (via QR) and sites left_site + 2 .. L - 1
    are right-orthonormal (via SVD/LQ).
    '''
    # TODO 11: Sweep left-to-right from 0 to left_site - 1 using QR decomposition and pushing R into site + 1;
    #          sweep right-to-left from L - 1 down to left_site + 2 using SVD/RQ and pushing U*S into site - 1.
    raise NotImplementedError("Complete TODO 11: canonicalize_mps_mixed_gauge")


def build_heisenberg_xxz_mpo(
    n_sites: int, J_xy: float = 1.0, J_z: float = 1.0, h_z: float = 0.0
) -> list[np.ndarray]:
    '''
    TODO 12: Construct the list of n_sites rank-4 MPO tensors W[site] of shape
    (b_left, b_right, d_phys, d_phys) for the 1D spin-1/2 XXZ chain.
    '''
    # TODO 12: Define 2x2 spin-1/2 operators I, Sp, Sm, Sz, populate the (5, 5, 2, 2) W_bulk tensor,
    #          and return [W_bulk[4:5, :, :, :], W_bulk, ..., W_bulk[:, 0:1, :, :]].
    raise NotImplementedError("Complete TODO 12: build_heisenberg_xxz_mpo")

## Part 4: Two-Site Variational DMRG Environments, Effective Hamiltonian & Sweeps (TODOs 13–16)

### Mathematical Specifications
1. **Left and Right MPO Environment Updates (`update_left_and_right_mpo_environments` — TODO 13)**:
   Maintain left environment tensors $L^{[\ell]}_{\alpha_\ell, b_\ell, \alpha'_\ell}$ (where $\alpha_\ell$ indexes the bra MPS $A^*$, $b_\ell$ indexes the MPO $W$, and $\alpha'_\ell$ indexes the ket MPS $A$) and right environment tensors $R^{[\ell]}_{\alpha_\ell, b_\ell, \alpha'_\ell}$:
   $$\begin{aligned}
   L_{\text{next}}(\alpha_R, b_R, \alpha'_R) &= \sum_{\alpha_L, b_L, \alpha'_L} \sum_{s, s'} L(\alpha_L, b_L, \alpha'_L)\,A^*_{\alpha_L, s, \alpha_R}\,W_{b_L, b_R, s, s'}\,A_{\alpha'_L, s', \alpha'_R} \\
   R_{\text{prev}}(\alpha_L, b_L, \alpha'_L) &= \sum_{\alpha_R, b_R, \alpha'_R} \sum_{s, s'} A^*_{\alpha_L, s, \alpha_R}\,W_{b_L, b_R, s, s'}\,A_{\alpha'_L, s', \alpha'_R}\,R(\alpha_R, b_R, \alpha'_R)
   \end{aligned}$$

2. **Two-Site Effective Hamiltonian Optimization (`optimize_two_site_wavefunction` — TODO 14)**:
   At bond $(\ell, \ell+1)$, the effective Hamiltonian acts on the two-site tensor $\Theta_{\alpha_L, s_\ell, s_{\ell+1}, \alpha_R}$ as:
   $$\mathbf{H}_{\text{eff}} = \text{\texttt{np.einsum("abc,bdik,dejl,feg->aijcfklg", L\_env, W\_left, W\_right, R\_env)}}$$
   Reshaping $\mathbf{H}_{\text{eff}}$ into a $(\chi_L d^2 \chi_R, \chi_L d^2 \chi_R)$ Hermitian matrix and finding its lowest eigenpair $(E_0, \Theta_0)$ via `scipy.linalg.eigh` variationally minimizes $\langle \Psi | \hat{H} | \Psi \rangle / \langle \Psi | \Psi \rangle$!

3. **Two-Site SVD Truncation (`split_and_truncate_two_site_tensor` — TODO 15)** & **Full DMRG Sweep Driver (`run_two_site_dmrg_sweeps` — TODO 16)**:
   Split $\Theta_0$ via SVD into `(A_left, A_right)` (pushing $\mathbf{S}\mathbf{V}^\dagger$ right during a left-to-right sweep, or pushing $\mathbf{U}\mathbf{S}$ left during a right-to-left sweep), update the MPO environments, and sweep back and forth until $E_0$ converges to `< tol`!

In [ ]:
def update_left_mpo_environment(
    L_env: np.ndarray, A_site: np.ndarray, W_site: np.ndarray
) -> np.ndarray:
    '''
    TODO 13a: Contract left environment L_env (alpha_L, b_L, alpha'_L) with bra A_site.conj()
    (alpha_L, s, alpha_R), MPO W_site (b_L, b_R, s, s'), and ket A_site (alpha'_L, s', alpha'_R)
    to return L_next of shape (alpha_R, b_R, alpha'_R).
    '''
    # TODO 13a: Use np.einsum(..., optimize=True) to contract L_env, A_site.conj(), W_site, and A_site.
    raise NotImplementedError("Complete TODO 13a: update_left_mpo_environment")


def update_right_mpo_environment(
    R_env: np.ndarray, A_site: np.ndarray, W_site: np.ndarray
) -> np.ndarray:
    '''
    TODO 13b: Contract right environment R_env (alpha_R, b_R, alpha'_R) with bra A_site.conj()
    (alpha_L, s, alpha_R), MPO W_site (b_L, b_R, s, s'), and ket A_site (alpha'_L, s', alpha'_R)
    to return R_prev of shape (alpha_L, b_L, alpha'_L).
    '''
    # TODO 13b: Use np.einsum(..., optimize=True) to contract A_site.conj(), W_site, A_site, and R_env.
    raise NotImplementedError("Complete TODO 13b: update_right_mpo_environment")


def optimize_two_site_wavefunction(
    L_env: np.ndarray, W_left: np.ndarray, W_right: np.ndarray, R_env: np.ndarray
) -> tuple[float, np.ndarray]:
    '''
    TODO 14: Construct the two-site effective Hamiltonian H_eff from L_env, W_left, W_right, R_env,
    diagonalize it with scipy.linalg.eigh, and return (e0, theta_ground_4d) where theta_ground_4d
    has shape (chi_L, d_left, d_right, chi_R).
    '''
    # TODO 14: Contract L_env, W_left, W_right, R_env into an 8-index tensor (a, i, j, c, f, k, l, g),
    #          reshape into a (dim_2site, dim_2site) matrix, diagonalize with eigh, and return (evals[0], evecs[:,0].reshape(...)).
    raise NotImplementedError("Complete TODO 14: optimize_two_site_wavefunction")


def split_and_truncate_two_site_tensor(
    theta_4d: np.ndarray,
    max_bond_dim: int = 16,
    svd_cutoff: float = 1e-12,
    direction: str = "right",
) -> tuple[np.ndarray, np.ndarray, float, float]:
    '''
    TODO 15: Split a two-site tensor theta_4d of shape (chi_L, d1, d2, chi_R) via SVD,
    truncate to max_bond_dim and svd_cutoff, and return (A_left, A_right, s_vn_entropy, discarded_weight).
    If direction == "right", A_left is left-orthonormal (U) and A_right absorbs diag(S) @ Vh.
    If direction == "left", A_left absorbs U @ diag(S) and A_right is right-orthonormal (Vh).
    '''
    # TODO 15: Reshape theta_4d to (chi_L * d1, d2 * chi_R), run svd(..., full_matrices=False),
    #          compute discarded_weight = sum(s[keep:]^2), normalize kept singular values,
    #          compute s_vn_entropy, and form (A_left, A_right) according to `direction`.
    raise NotImplementedError("Complete TODO 15: split_and_truncate_two_site_tensor")


def run_two_site_dmrg_sweeps(
    mpo_tensors: list[np.ndarray],
    n_sites: int,
    max_bond_dim: int = 16,
    n_sweeps: int = 4,
    tol: float = 1e-10,
) -> dict[str, float | list[float] | list[np.ndarray]]:
    '''
    TODO 16: Execute two-site variational DMRG sweeps (left-to-right and right-to-left)
    using TODOs 11-15 to find the ground state of `mpo_tensors`.
    Return a dict with keys:
        'e_dmrg': float
        'sweep_energies': list[float]
        'half_chain_entropy': float
        'max_discarded_weight': float
        'mps_tensors': list[np.ndarray]
    '''
    # TODO 16: Initialize a normalized MPS and right environments R_envs[1..L-1] (with L_envs[0] = R_envs[L] = ones(1,1,1)),
    #          sweep left-to-right (bonds 0..L-2, direction="right") and right-to-left (bonds L-2..0, direction="left"),
    #          and return the converged ground-state energy and MPS.
    raise NotImplementedError("Complete TODO 16: run_two_site_dmrg_sweeps")

## Part 5: Step-by-Step Checkpoint & Verification Suite

Run `run_section05b_checkpoints()` below as you implement each `# TODO` function. It verifies all 9 Bootstrap Embedding functions on a 4-orbital correlated ring/chain model and all 7 DMRG/MPS functions on a 6-site spin-$1/2$ Heisenberg XXZ chain against exact diagonalization!

In [2]:
def run_section05b_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 16 #TODO functions in Section 05b (BE + DMRG) using PySCF.'''
    passed = 0
    total_todos = 16

    # Also build a 2-electron H4^2+ model in PySCF for testing 2-electron fragment FCI matching
    mol_h4_2e = gto.M(
        atom="H 0.0 0.0 0.0; H 0.0 0.0 1.5; H 1.5 0.0 1.5; H 1.5 0.0 0.0",
        unit="Bohr",
        basis="sto-3g",
        charge=2,
        spin=0,
        verbose=0,
    )
    mf_h4_2e = scf.RHF(mol_h4_2e).run(conv_tol=1e-12)
    C_occ_lo_2e = X_lowdin_inv @ mf_h4_2e.mo_coeff[:, :1]
    P_hf_lo_h4 = 2.0 * (C_occ_lo_h4 @ C_occ_lo_h4.T)

    try:
        # --- PART 1 & 2 CHECKS: BOOTSTRAP EMBEDDING (TODOs 1-9) ---
        sch = partition_c_occ_and_schmidt_svd(C_occ_lo_h4, frag_indices=[0, 1])
        assert sch["n_bath"] == 2 and sch["U_bath"].shape == (2, 2)
        passed += 1
        print("[PASS] TODO 1 : partition_c_occ_and_schmidt_svd")

        T_emb, P_env_lo = assemble_embedding_rotation_and_env_density(
            4, [0, 1], sch["comp_indices"], sch["U_bath"], sch["U_env"]
        )
        assert T_emb.shape == (4, 4) and np.linalg.norm(T_emb.T @ T_emb - np.eye(4)) < 1e-12
        passed += 1
        print("[PASS] TODO 2 : assemble_embedding_rotation_and_env_density")

        h_emb, V_emb = project_hamiltonian_to_embedding_space(h_lo_h4, V_lo_h4, T_emb, P_env_lo)
        assert h_emb.shape == (4, 4) and V_emb.shape == (4, 4, 4, 4)
        passed += 1
        print("[PASS] TODO 3 : project_hamiltonian_to_embedding_space")

        s_1rdm, T_1rdm = extract_1rdm_schmidt_bath(P_hf_lo_h4, [0, 1])
        assert len(s_1rdm) == 2 and T_1rdm.shape == (4, 4)
        passed += 1
        print("[PASS] TODO 4 : extract_1rdm_schmidt_bath")

        sch_2e = partition_c_occ_and_schmidt_svd(C_occ_lo_2e, [0, 1])
        T_2e, P_env_2e = assemble_embedding_rotation_and_env_density(
            4, [0, 1], sch_2e["comp_indices"], sch_2e["U_bath"], sch_2e["U_env"]
        )
        h_emb_2e, V_emb_2e = project_hamiltonian_to_embedding_space(h_lo_h4, V_lo_h4, T_2e, P_env_2e)
        e_fci_d, rdm1_c, rdm2_c = solve_dressed_fragment_fci(h_emb_2e, V_emb_2e, [0], [1], mu_global=0.0)
        assert np.isclose(np.trace(rdm1_c), 2.0, atol=1e-12)
        passed += 1
        print("[PASS] TODO 5 : solve_dressed_fragment_fci")

        D_lo_2e = C_occ_lo_2e @ C_occ_lo_2e.T
        P_hf_emb_2e = T_2e.T @ (2.0 * D_lo_2e) @ T_2e
        dp, k2 = compute_two_body_cumulant_and_delta_p(rdm1_c, rdm2_c, P_hf_emb_2e, use_be_approx=True)
        assert dp.shape == rdm1_c.shape and k2.shape == rdm2_c.shape
        passed += 1
        print("[PASS] TODO 6 : compute_two_body_cumulant_and_delta_p")

        J_lo_2e = np.einsum("uvls,ls->uv", V_lo_h4, D_lo_2e)
        K_lo_2e = np.einsum("ulvs,ls->uv", V_lo_h4, D_lo_2e)
        F_hf_emb_2e = T_2e.T @ (h_lo_h4 + 2.0 * J_lo_2e - K_lo_2e) @ T_2e
        de_frag = compute_fragment_cumulant_energy_contribution([0], F_hf_emb_2e, V_emb_2e, dp, k2)
        assert de_frag < 0.0
        passed += 1
        print(f"[PASS] TODO 7 : compute_fragment_cumulant_energy_contribution (dE_A = {de_frag:+.6f} Eh)")

        res_vec = evaluate_be_matching_residuals([rdm1_c, rdm1_c], [(0, 1, 1, 0)], [[0], [0]], target_n_elec=2)
        assert res_vec.shape == (2,)
        passed += 1
        print("[PASS] TODO 8 : evaluate_be_matching_residuals")

        frags = [[3, 0, 1], [0, 1, 2], [1, 2, 3], [2, 3, 0]]
        ctrs = [[0], [1], [2], [3]]
        be_res = run_bootstrap_embedding_newton_loop(h_lo_h4, V_lo_h4, float(mol_h4_2e.energy_nuc()), 2, frags, ctrs)
        assert be_res["final_rms_mismatch"] < 1e-8 and np.isclose(be_res["total_center_electrons"], 2.0, atol=1e-7)
        passed += 1
        print(f"[PASS] TODO 9 : run_bootstrap_embedding_newton_loop (E_BE = {be_res['e_be_total']:.8f} Eh, mismatch = {be_res['final_rms_mismatch']:.2e})")

        # --- PART 3 & 4 CHECKS: MPS & TWO-SITE DMRG (TODOs 10-16) ---
        L_sites = 6
        mpo_xxz = build_heisenberg_xxz_mpo(L_sites, J_xy=1.0, J_z=1.0, h_z=0.1)
        H_exact_mpo = mpo_xxz[0]
        for site in range(1, L_sites):
            H_exact_mpo = np.einsum("ab...,bcjk->ac...jk", H_exact_mpo, mpo_xxz[site])
        bra_axes = [2 + 2 * k for k in range(L_sites)]
        ket_axes = [2 + 2 * k + 1 for k in range(L_sites)]
        H_mat_exact = H_exact_mpo[0, 0].transpose(*(bra_axes + ket_axes)).reshape(2**L_sites, 2**L_sites)
        evals_ex, evecs_ex = eigh(H_mat_exact)
        e_exact_xxz = float(evals_ex[0])
        psi_exact_xxz = evecs_ex[:, 0]

        mps_from_psi, entropies = wavefunction_to_mps_svd(psi_exact_xxz, n_sites=L_sites, max_bond_dim=16)
        assert len(mps_from_psi) == L_sites and len(entropies) == L_sites - 1 and entropies[2] > 0.1
        passed += 1
        print(f"[PASS] TODO 10: wavefunction_to_mps_svd (Half-chain S_vN = {entropies[2]:.5f})")

        mps_mixed = canonicalize_mps_mixed_gauge(mps_from_psi, left_site=2)
        A0 = mps_mixed[0].reshape(-1, mps_mixed[0].shape[2])
        AL = mps_mixed[-1].reshape(mps_mixed[-1].shape[0], -1)
        assert np.linalg.norm(A0.conj().T @ A0 - np.eye(A0.shape[1])) < 1e-12
        assert np.linalg.norm(AL @ AL.conj().T - np.eye(AL.shape[0])) < 1e-12
        passed += 1
        print("[PASS] TODO 11: canonicalize_mps_mixed_gauge")

        assert len(mpo_xxz) == L_sites and mpo_xxz[0].shape == (1, 5, 2, 2) and mpo_xxz[1].shape == (5, 5, 2, 2)
        passed += 1
        print("[PASS] TODO 12: build_heisenberg_xxz_mpo")

        L0 = np.ones((1, 1, 1), dtype=np.float64)
        L1 = update_left_mpo_environment(L0, mps_mixed[0], mpo_xxz[0])
        R6 = np.ones((1, 1, 1), dtype=np.float64)
        R5 = update_right_mpo_environment(R6, mps_mixed[-1], mpo_xxz[-1])
        assert L1.shape[1] == 5 and R5.shape[1] == 5
        passed += 1
        print("[PASS] TODO 13: update_left/right_mpo_environment")

        R_list = [None] * (L_sites + 1)
        R_list[L_sites] = np.ones((1, 1, 1))
        for k in range(L_sites - 1, 1, -1):
            R_list[k] = update_right_mpo_environment(R_list[k + 1], mps_mixed[k], mpo_xxz[k])
        e_2site, theta_01 = optimize_two_site_wavefunction(L0, mpo_xxz[0], mpo_xxz[1], R_list[2])
        assert np.isclose(e_2site, e_exact_xxz, atol=1e-8)
        passed += 1
        print(f"[PASS] TODO 14: optimize_two_site_wavefunction (E_0 = {e_2site:.8f} vs Exact {e_exact_xxz:.8f})")

        A_l, A_r, s_vn_b, disc_w = split_and_truncate_two_site_tensor(theta_01, max_bond_dim=16, direction="right")
        assert A_l.shape[2] == A_r.shape[0] and disc_w < 1e-12
        passed += 1
        print("[PASS] TODO 15: split_and_truncate_two_site_tensor")

        dmrg_res = run_two_site_dmrg_sweeps(mpo_xxz, n_sites=L_sites, max_bond_dim=16, n_sweeps=4)
        assert np.isclose(dmrg_res["e_dmrg"], e_exact_xxz, atol=1e-8)
        passed += 1
        print(f"[PASS] TODO 16: run_two_site_dmrg_sweeps (E_DMRG = {dmrg_res['e_dmrg']:.10f}, Exact = {e_exact_xxz:.10f})")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 05b (BE + DMRG) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section05b_checkpoints()
print("Section 05b checkpoint suite loaded. Implement TODOs 1-16 and call `run_section05b_checkpoints()`!")

Section 05b checkpoint suite loaded. Implement TODOs 1-16 and call `run_section05b_checkpoints()`!
